# 02. Turn the approved joints into uniformly masked tokens

A token is one joint at one time. This lesson fixes the anatomical indexing, draws uniform masks, and distinguishes random completion from prediction of the future. The mask never consults motion intensity or velocity.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Fix the deduplicated anatomy

The user supplied feature requirements rather than a literal landmark list. We derive an explicit 16-joint union from exp5's lower-limb geometry and the declared arm/foot extension. These are the only joints that enter the model or its masking universe.

# The exact joint contract

The user brief names features and asks for deduplication, but does not supply a literal list of keypoints. This project therefore declares a transparent feature-derived set. We start with the hip, knee, and ankle geometry used in exp5, then include arm swing, elbow flexion, and foot-axis orientation. Their union has **16 unique BlazePose landmarks**. This is a study choice, not an anatomical set mandated by the S-JEPA paper.

|BLAZEPOSE_33 index|Keypoint name|Features involved|
|---:|---|---|
|11|Left shoulder|Left hip angle; left elbow angle; context torso scale|
|12|Right shoulder|Right hip angle; right elbow angle; context torso scale|
|13|Left elbow|Left elbow angle|
|14|Right elbow|Right elbow angle|
|15|Left wrist|Left elbow angle; left arm swing speed|
|16|Right wrist|Right elbow angle; right arm swing speed|
|23|Left hip|Left hip and knee angles; pelvis center; context torso scale|
|24|Right hip|Right hip and knee angles; pelvis center; context torso scale|
|25|Left knee|Left hip, knee, and ankle angles|
|26|Right knee|Right hip, knee, and ankle angles|
|27|Left ankle|Left knee and ankle angles; ankle separation|
|28|Right ankle|Right knee and ankle angles; ankle separation|
|29|Left heel|Left foot-axis inclination|
|30|Right heel|Right foot-axis inclination|
|31|Left foot index|Left ankle angle; left foot-axis inclination|
|32|Right foot index|Right ankle angle; right foot-axis inclination|

`worldmodels.data.JOINT_IDS` is the numerical source of truth. After selection, tensor position 0 means BlazePose joint 11, and tensor position 15 means BlazePose joint 32. Tensor position 11 is **not** BlazePose joint 11. Model configuration and every mask must preserve this mapping. Neither the face nor finger landmarks enter the predictor or its masking universe.

A token is one joint in one frame. A 16-frame window has 256 candidate positions. Missing observations retain a separate Boolean validity mask. Uniform masking draws from eligible observed positions without consulting velocity, motion magnitude, condition labels, or confidence magnitude. Confidence only determines whether an observation is valid. It never makes an observed fast joint more likely to be masked.

In [ ]:
from worldmodels.data import JOINT_IDS
from worldmodels.masking import JOINT_IDS as MASK_JOINT_IDS
assert JOINT_IDS == MASK_JOINT_IDS
assert list(JOINT_IDS) == sorted(set(JOINT_IDS))
print("Tensor slot -> BlazePose ID:", dict(enumerate(JOINT_IDS)))

## Step 2. Give shape a physical meaning

`[batch, time, joint, coordinate]` is the tensor order. A 16-frame window and 16 joints give 256 possible joint-time tokens. The last dimension has x and y. Visibility is a separate Boolean array and does not create another coordinate dimension. Index 0 in the reduced tensor means BlazePose index 11, not the nose.

In [ ]:
from worldmodels.notebook_support import lesson_data
from worldmodels.masking import uniform_mask, forecast_mask
fixture = lesson_data(n=4)
x = torch.as_tensor(fixture["x"])
valid = torch.as_tensor(fixture["valid"])
print("coordinates:", tuple(x.shape), "validity:", tuple(valid.shape))
if not REAL:
    valid[0, 3, 5] = False  # synthetic missingness exercise; real mode preserves measured validity

In [ ]:
display(SVG(filename=str(PROJECT / "images/uniform_mask.svg")))

## Step 3. Draw a uniform mask without replacement

A `True` target flag means the model must predict that token. The routine samples a fixed-size subset of valid positions uniformly. Missing positions are excluded from both targets and visible context. Every example retains at least one context and one target.

MAMP's main published strategy favors stronger motion. This study intentionally follows the requested uniform variant. Random numbers inside a motion-weighted sampler would not make its final sampling distribution uniform.

In [ ]:
mask = uniform_mask(valid, ratio=0.5, generator=torch.Generator().manual_seed(27))
assert not (mask & ~valid).any()
print("Targets per window:", mask.flatten(1).sum(1).tolist())
fig, ax = plt.subplots(figsize=(9,4))
ax.imshow(mask[0].T, aspect="auto", interpolation="nearest", cmap="Blues")
ax.set(yticks=np.arange(16), yticklabels=JOINT_IDS, xlabel="Frame slot", ylabel="BlazePose joint ID", title="Blue cells are prediction targets")
plt.show()

## Step 4. Verify independence from movement

The masking function accepts validity and a random generator, but no coordinates. Multiplying every displacement by a thousand cannot change the chosen positions when validity and the seed are fixed. This test checks the intended causal dependency of the software rather than simply inspecting a random-looking plot.

In [ ]:
changed_motion = x * 1000
again = uniform_mask(valid, ratio=0.5, generator=torch.Generator().manual_seed(27))
assert torch.equal(mask, again)
print("Changing motion values has no input path into mask selection.")

## Step 5. Distinguish completion from forecasting

Random masks can leave visible joints before and after a target time. A completion model may therefore use both temporal directions. Forecasting requires a different mask: all later observations are hidden from the online encoder, including joints that would otherwise be easy to reveal. The teacher can see the later outcomes only when building training or retrospective scoring targets.

In [ ]:
future = forecast_mask(valid, context_frames=8)
assert not future[:, :8].any()
assert torch.equal(future[:, 8:], valid[:, 8:])
fig, ax = plt.subplots(figsize=(9,3))
ax.imshow(future[0].T, aspect="auto", interpolation="nearest", cmap="Oranges")
ax.set(xlabel="Frame slot", ylabel="Joint slot", title="Future-only targets: frames 8 through 15")
plt.show()

## Step 6. Keep masks consistent across controlled variants

A positional-noise or latent-variable experiment should reuse the same eligible inputs, seed, source split, and masking rule. Otherwise an apparent stochasticity benefit may reflect easier masks or a different cohort. Never add face or finger landmarks just because a new code path expects 33 coordinates.

In [ ]:
print("Allowed targets only:", JOINT_IDS)
print("Example token targeted?", bool(mask[0,3,5]))
if REAL:
    from worldmodels.training import load_checkpoint
    saved_encoder = load_checkpoint(comparison_path() / "normal_completion.pt")
    with torch.no_grad():
        actual_completion = saved_encoder(x, valid, mask)
    print("Actual trained-model masked latent prediction:", tuple(actual_completion["latent"].shape))

In [ ]:
lesson_evidence = persist_lesson("02_joint_tokens_and_uniform_masks", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.